# Nested CV primjer na Adult (income) skupu podataka

- Unutarnja CV: GridSearchCV (tuning hiperparametara)
- Vanjska CV: StratifiedKFold (nepristrana procjena nakon tuninga)

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [3]:
# 1) Učitaj Adult (preko scikit-learn-a)
adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data
y = adult.target

# Target je string ('<=50K', '>50K'); pretvorimo u 0/1
y = (y == ">50K").astype(int)

# 2) Podjela kolona po tipu
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# 3) Pretprocesiranje:
#    - numeričke: imputacija medianom + skaliranje
#    - kategorijske: imputacija najčešćom + OneHotEncoder (drop='first' + handle_unknown)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

# 4) Cijeli pipeline: preprocesor + klasifikator
pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(
        solver="liblinear",            # stabilno za binarni problem
        max_iter=1000
    ))
])

# 5) Grid za tuning unutar unutarnje CV
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__class_weight": [None, "balanced"]
}

# 6) Unutarnja i vanjska CV
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)   # za GridSearchCV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # za nested evaluaciju

# 7) GridSearchCV (unutarnja CV)
gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1
)

# 8) Brza nested evaluacija: cross_val_score nad GridSearchCV (vanjska CV)
outer_scores = cross_val_score(gs, X, y, cv=outer_cv, scoring="roc_auc", n_jobs=-1)
print(f"Nested CV AUC: {outer_scores.mean():.4f} +/- {outer_scores.std():.4f}")

# 9) (Opcionalno) Ručni prikaz po vanjskom foldu – da se vidi da se tuning radi na trainu, a test je netaknut
#    Ovo ispisuje best parametre po foldu i AUC na vanjskom testu.
print("\nDetalji po vanjskom foldu:")
fold_aucs = []
for i, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    # Fit unutarnjeg GridSearch-a SAMO na vanjskom trainu
    gs.fit(X_tr, y_tr)

    # Predikcija na vanjskom testu (tek sada ga "vidimo")
    y_proba = gs.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, y_proba)
    fold_aucs.append(auc)

    print(f"Fold {i}: best_params={gs.best_params_}, inner_CV_AUC={gs.best_score_:.4f}, outer_test_AUC={auc:.4f}")

print(f"\nProsječni outer AUC (ručno): {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}")

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1020)>